# NBA 3.1 HCP Universe — Data Exploration

This notebook loads and profiles the source data used to construct the Kerendia HCP Universe.

**Objectives:**
1. Load `lrx_nbrx` claims data and apply the 24-month lookback window
2. Profile distributions: specialty, indication, product class, NBRX volume
3. Load and summarize Kerendia writer data (C2 source)
4. Load and summarize the NBA 3.0 baseline universe
5. Load the ID mapping table

In [0]:
# ============================================================
# Imports & Parameters
# ============================================================
import builtins
from functools import reduce
from pyspark.sql import functions as F, Window

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA  = "pace_ker_in_house_schema"
SOURCE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.lrx_nbrx"

# 24-month lookback
LOOKBACK_START = "2024-10-01"
LOOKBACK_END   = "2026-09-30"

# Indication x Drug Pool configuration
INDICATIONS = ["T2D", "CKD", "HF", "T1D"]
DRUG_POOLS = {
    "SGLT2_GLP1": ["SGLT2", "GLP1"],
    "ACE_ARB":    ["ACE", "ARB"],
}

# Column names
INDICATION_COL = "INDICATION_GROUP_APP1"
PRODUCT_COL    = "PRODUCT_CLASS"
METRIC_COL     = "NBRX"

# Kerendia writer data path
KERENDIA_NBRX_PATH = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/"
    "kerendia_nbrx_from_raw_fct_rx"
)

# NBA 3.0 baseline
NBA30_PATH = (
    f"/Volumes/{CATALOG}/"
    "kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
    "dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"
)

print("Parameters loaded.")
print(f"  Source: {SOURCE_TABLE}")
print(f"  Lookback: {LOOKBACK_START} – {LOOKBACK_END}")
print(f"  Indications: {INDICATIONS}")
print(f"  Drug pools: {list(DRUG_POOLS.keys())}")

In [0]:
# ============================================================
# Load lrx_nbrx & apply lookback window
# ============================================================
df_src = spark.table(SOURCE_TABLE)
lrx = df_src.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) &
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)

print(f"Full table rows:     {df_src.count():,}")
print(f"Lookback rows:       {lrx.count():,}")
print(f"Unique HCPs:         {lrx.select('PROVIDER_ID').distinct().count():,}")
print(f"\nSchema:")
lrx.printSchema()

In [0]:
# ============================================================
# Verify date range of lookback data
# ============================================================
date_range = lrx.agg(
    F.min("WEEK_ENDING_DATE").alias("min_date"),
    F.max("WEEK_ENDING_DATE").alias("max_date"),
    F.countDistinct("WEEK_ENDING_DATE").alias("n_weeks")
).collect()[0]

print(f"Date range: {date_range['min_date']} to {date_range['max_date']}")
print(f"Distinct weeks: {date_range['n_weeks']}")

In [0]:
# ============================================================
# SPECIALTY_GROUP distribution
# ============================================================
spec_dist = (
    lrx.groupBy("SPECIALTY_GROUP")
    .agg(
        F.countDistinct("PROVIDER_ID").alias("N_HCPS"),
        F.sum(METRIC_COL).alias("TOTAL_NBRX"),
    )
    .orderBy(F.desc("N_HCPS"))
)
display(spec_dist)

In [0]:
# ============================================================
# PRODUCT_CLASS distribution (ranked by NBRX volume)
# ============================================================
pc_dist = (
    lrx.groupBy(PRODUCT_COL)
    .agg(
        F.sum(METRIC_COL).alias("TOTAL_NBRX"),
        F.countDistinct("PROVIDER_ID").alias("N_HCPS"),
    )
    .orderBy(F.desc("TOTAL_NBRX"))
)
display(pc_dist)

In [0]:
# ============================================================
# HCP counts per indication (using .contains() for compound values)
# ============================================================
for ind in INDICATIONS:
    ind_ct = lrx.filter(F.col(INDICATION_COL).contains(ind)).select("PROVIDER_ID").distinct().count()
    print(f"{ind}: {ind_ct:,} unique HCPs")

# Cross-tab: indication x drug pool
print(f"\n{'Indication':<10} {'SGLT2/GLP1':>12} {'ACE/ARB':>12}")
print("-" * 36)
for ind in INDICATIONS:
    counts = []
    for pool_name, pool_products in DRUG_POOLS.items():
        ct = (
            lrx.filter(
                (F.col(INDICATION_COL).contains(ind)) &
                (F.col(PRODUCT_COL).isin(pool_products))
            )
            .select("PROVIDER_ID").distinct().count()
        )
        counts.append(ct)
    print(f"{ind:<10} {counts[0]:>12,} {counts[1]:>12,}")

In [0]:
# ============================================================
# NBRX volume distribution per HCP (overall)
# ============================================================
hcp_vol = lrx.groupBy("PROVIDER_ID").agg(F.sum(METRIC_COL).alias("TOTAL_NBRX"))

print("HCP-level NBRX volume summary:")
hcp_vol.describe("TOTAL_NBRX").show()

# Volume buckets
hcp_vol = hcp_vol.withColumn(
    "VOLUME_BUCKET",
    F.when(F.col("TOTAL_NBRX") <= 1, "0-1")
    .when(F.col("TOTAL_NBRX") <= 5, "2-5")
    .when(F.col("TOTAL_NBRX") <= 20, "6-20")
    .when(F.col("TOTAL_NBRX") <= 100, "21-100")
    .when(F.col("TOTAL_NBRX") <= 500, "101-500")
    .otherwise("500+")
)
bucket_dist = hcp_vol.groupBy("VOLUME_BUCKET").count().orderBy("VOLUME_BUCKET")
display(bucket_dist)

In [0]:
# ============================================================
# Load Kerendia Writers (from Raw FCT_RX)
# ============================================================
ker_fct_rx = spark.read.format("delta").load(KERENDIA_NBRX_PATH)
ker_fct_rx = ker_fct_rx.withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string"))

print(f"Kerendia FCT_RX rollup: {ker_fct_rx.count():,} HCPs")
ker_fct_rx.printSchema()

# Writer counts by different definitions
nbrx_all = ker_fct_rx.filter(F.col("NBRX_KERENDIA_OVERALL") > 0).count()
nbrx_paid = ker_fct_rx.filter(F.col("NBRX_KERENDIA_PAID") > 0).count() if "NBRX_KERENDIA_PAID" in ker_fct_rx.columns else "N/A"
trx_all = ker_fct_rx.filter(F.col("TRX_KERENDIA_OVERALL") > 0).count()

print(f"\nKerendia Writer Counts:")
print(f"  NBRX > 0 (all claims):  {nbrx_all:,}")
print(f"  NBRX > 0 (paid only):   {nbrx_paid}")
print(f"  TRX > 0  (all claims):  {trx_all:,}")

# C2 definition: NBRX_KERENDIA_OVERALL > 0
ker_writers = ker_fct_rx.filter(F.col("NBRX_KERENDIA_OVERALL") > 0).select("PROVIDER_ID")
print(f"\n>>> C2 Kerendia Writers: {ker_writers.count():,}")

In [0]:
# ============================================================
# Load NBA 3.0 Baseline Universe
# ============================================================
nba30 = spark.read.format("delta").load(NBA30_PATH)
nba30_ct = nba30.count()

print(f"NBA 3.0 Universe: {nba30_ct:,} HCPs")
nba30.printSchema()

# Flag distribution
for flag in ["KERENDIA_FLAG", "SGLT2_GLP1_FLAG", "NEPH_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG"]:
    if flag in nba30.columns:
        ct = nba30.filter(F.col(flag) == 1).count()
        print(f"  {flag} = 1: {ct:,} ({100*ct/nba30_ct:.1f}%)")

In [0]:
# ============================================================
# Load ID Mapping Table
# ============================================================
MAPPING_TABLE = f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"
id_map = spark.table(MAPPING_TABLE)

print(f"ID Mapping rows: {id_map.count():,}")
print(f"Unique PROVIDER_IDs: {id_map.select('PROVIDER_ID').distinct().count():,}")
print(f"Unique BH_IDs: {id_map.select('BH_ID').distinct().count():,}")
id_map.printSchema()

In [0]:
# ============================================================
# Kerendia writers visible in lrx_nbrx (paid NBRX, 24-mo)
# This is M2 — the in-table Kerendia signal
# ============================================================
m2_writers = (
    lrx.filter(
        (F.col(PRODUCT_COL) == "KERENDIA") &
        (F.col(METRIC_COL) > 0)
    )
    .select("PROVIDER_ID").distinct()
)
m2_ct = m2_writers.count()

print(f"Kerendia writers in lrx_nbrx (paid NBRX>0, 24-mo): {m2_ct:,}")
print(f"C2 writers (FCT_RX, all claims, full history):       {ker_writers.count():,}")
print(f"\nDelta: {ker_writers.count() - m2_ct:,} writers visible only via C2 (non-paid / older claims)")

## Summary

| Data Source | Records / HCPs | Notes |
|:------------|:---------------|:------|
| `lrx_nbrx` (full) | ~247M rows | Full claims table |
| `lrx_nbrx` (24-mo lookback) | ~104M rows, ~1.36M HCPs | Oct 2024 – Sep 2026 |
| Kerendia Writers (C2) | 61,583 | NBRX > 0, all claim types, full history |
| NBA 3.0 Baseline | 232,965 | Prior universe for overlap validation |
| ID Mapping | ~1.65M rows | PROVIDER_ID ↔ BH_ID ↔ NPI |

**Key insight:** Only ~40K Kerendia writers are visible in `lrx_nbrx` (paid NBRX, 24-mo). The remaining ~21K require C2 (non-paid claims or older history).

**Next:** `02_Universe_Construction` — build C1–C4 criteria and assemble the universe.